# DOFA — Tensor Shapes & Datenfluss

Dieses Notebook verfolgt einen S2-Input (12 Bänder, m-cashew-plantation) Schritt für Schritt durch DOFA:

```
Bild [B, 12, 224, 224]
        │
        ▼
┌──────────────────────────────────────────┐
│  Hypernetzwerk (Dynamic_MLP_OFA)         │
│  Wavelengths → Conv-Kernel               │
│  [12] → sincos → FCRes → WeightGen       │
│                        → Kernel [768,12,16,16]
└──────────────────────────────────────────┘
        │  conv2d(Bild, Kernel)
        ▼
  Patch-Tokens [B, 196, 768]
        │  + pos_embed + CLS
        ▼
  12× Transformer Block [B, 197, 768]
        │
        ▼
  4 Outputs (Layer 3,5,7,11) [B, 768, 14, 14]
```

In [ ]:
import sys
sys.path.insert(0, "/p/project1/hai_1280/pangaea-bench-dev-nc")

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pangaea.encoders.dofa_encoder import DOFA_Encoder
from pangaea.encoders.pos_embed import get_1d_sincos_pos_embed_from_grid_torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Setup: m-cashew-plantation Konfiguration
12 S2-Bänder, Bildgröße 256 px (wird auf Encoder-Größe 224 resized).

In [ ]:
# Bands aus mcashew-plantation.yaml
BANDS = ["B1","B2","B3","B4","B5","B6","B7","B8","B8A","B9","B11","B12"]
INPUT_BANDS = {"optical": BANDS}

# S2 Wellenlängen (µm) — aus dofa.yaml
WAVELENGTHS = {
    "B1":0.44, "B2":0.49, "B3":0.56, "B4":0.665, "B5":0.705,
    "B6":0.74, "B7":0.783, "B8":0.832, "B8A":0.864, "B9":0.945,
    "B10":1.373, "B11":1.61, "B12":2.20,
}
WAVE_LIST_FULL = {"optical": WAVELENGTHS, "sar": {"VV":3.75,"VH":3.75}}

wv_list = [WAVELENGTHS[b] for b in BANDS]
print(f"Anzahl Bänder : {len(BANDS)}")
print(f"Wellenlängen  : {wv_list}")

In [ ]:
encoder = DOFA_Encoder(
    encoder_weights="/p/project1/hai_1280/pangaea-bench-dev-nc/pretrained_models/DOFA_ViT_base_e100.pth",
    input_bands=INPUT_BANDS,
    input_size=224,
    embed_dim=768,
    output_dim=768,
    output_layers=[3, 5, 7, 11],
    wave_list=WAVE_LIST_FULL,
    download_url="",
    patch_size=16,
    depth=12,
    num_heads=12,
    mlp_ratio=4.0,
    use_norm=False,
).to(device)
encoder.eval()
print("wv_list:", encoder.wv_list)
print("Encoder geladen.")

## 2. Input-Tensor
Cashew-plantation: normalisierte S2-Reflektanzen, Batchgröße 2.

In [ ]:
B  = 2      # Batch
C  = len(BANDS)   # 12 Bänder
T  = 1      # single-temporal
H  = W = 224

# Simulierter normalisierter Input (mean≈0, std≈1)
# In der echten Pipeline: (pixel - data_mean) / data_std
torch.manual_seed(0)
raw_input = {"optical": torch.randn(B, C, T, H, W, device=device)}

print("=== Input-Tensor (wie er aus dem DataLoader kommt) ===")
print(f"  raw_input['optical'] : {tuple(raw_input['optical'].shape)}")
print(f"  Bedeutung            : (Batch={B}, Kanäle={C}, Zeit={T}, H={H}, W={W})")
print()

# Schritt 1: squeeze(2) entfernt die Zeitdimension
# Code in DOFA.forward: x = [image[m].squeeze(2) for m in self.input_bands.keys()]
x_squeezed = raw_input["optical"].squeeze(2)
print(f"  Nach squeeze(2)      : {tuple(x_squeezed.shape)}")
print(f"  Bedeutung            : (Batch={B}, Kanäle={C}, H={H}, W={W})")
print()

# Schritt 2: torch.cat über Modalitäten (hier nur optical)
# Code: x = torch.cat(x, dim=1)
x = x_squeezed   # keine Änderung bei single-modality
print(f"  Nach cat()           : {tuple(x.shape)}   (identisch, nur eine Modalität)")
print()

# Wellenlängen-Tensor
wavelist = torch.tensor(encoder.wv_list, device=device).float()
print(f"  wavelist             : {tuple(wavelist.shape)}   → {wavelist.tolist()}")

## 3. Hypernetzwerk: Wavelengths → Conv-Kernel

DOFA's Kernidee: statt einem **festen** Patch-Embedding-Kernel wird ein **dynamischer** Kernel aus den Wellenlängen erzeugt.

```
wavelist [12]  →  sincos embedding  →  [12, 128]
                       │
                  FCResLayer [12, 128]
                       │
          TransformerWeightGenerator
          ┌────────────────────────┐
          │ weight_tokens [128,128]│  (gelernt)
          │ + band_tokens  [12,128]│  (aus FCRes)
          │ + bias_token    [1,128]│  (gelernt)
          │ = Sequenz [141, 128]   │
          │       │                │
          │  Transformer (1 Layer) │
          │       │                │
          │  fc_weight → weights [12, 16·16·768]
          │  fc_bias   → bias    [768]
          └────────────────────────┘
               │
          reshape → Kernel [768, 12, 16, 16]
```

In [ ]:
patch_embed = encoder.patch_embed
wv_planes   = patch_embed.wv_planes   # 128
kernel_size = patch_embed.kernel_size # 16
embed_dim   = patch_embed.embed_dim   # 768
n_bands     = wavelist.size(0)        # 12

print("--- Schritt A: sincos Positional Embedding der Wellenlängen ---")
# Code: waves = get_1d_sincos_pos_embed_from_grid_torch(wv_planes, wvs * 1000)
waves_raw = get_1d_sincos_pos_embed_from_grid_torch(wv_planes, wavelist * 1000).float()
print(f"  Input  wavelist              : {tuple(wavelist.shape)}    (n_bands)")
print(f"  wavelist * 1000 (µm→nm)      : {(wavelist*1000).tolist()}")
print(f"  Output sincos embedding      : {tuple(waves_raw.shape)}  (n_bands, wv_planes=128)")
print()

print("--- Schritt B: FCResLayer (2× Linear + ReLU + Residual) ---")
# Code: waves = self.fclayer(waves)
with torch.no_grad():
    waves = patch_embed.fclayer(waves_raw)
print(f"  Input  : {tuple(waves_raw.shape)}")
print(f"  Output : {tuple(waves.shape)}   (Shape unverändert, Werte transformiert)")
print()

print("--- Schritt C: TransformerWeightGenerator ---")
wg = patch_embed.weight_generator
print(f"  weight_tokens (gelernt) : {tuple(wg.weight_tokens.shape)}   (128 'Lese'-Tokens)")
print(f"  bias_token    (gelernt) : {tuple(wg.bias_token.shape)}     (1 Bias-Token)")
print(f"  band_tokens (aus FCRes) : {tuple(waves.shape)}   (12 Band-Tokens)")
print()
# concat: [128 + 12 + 1, 128] = [141, 128]
pos_wave = waves.unsqueeze(1)  # transformer erwartet [seq_len, batch=1, dim] — hier kein batch
seq = torch.cat([wg.weight_tokens, waves, wg.bias_token], dim=0)
print(f"  Sequenz für Transformer  : {tuple(seq.shape)}")
print(f"  Aufbau: 128 weight_tokens + {n_bands} band_tokens + 1 bias_token = {128+n_bands+1}")
print()

with torch.no_grad():
    weights_flat, bias = wg(waves)
print(f"  Output weights (flat)   : {tuple(weights_flat.shape)}")
print(f"  Bedeutung               : {n_bands} Bänder × ({kernel_size}×{kernel_size} × {embed_dim}) = {n_bands}×{kernel_size*kernel_size*embed_dim}")
print(f"  Output bias             : {tuple(bias.shape)}")

In [ ]:
print("--- Schritt D: Reshape zum Conv-Kernel ---")
# Code in Dynamic_MLP_OFA.forward:
# dynamic_weight = weight.view(inplanes, kernel_size, kernel_size, embed_dim)
# dynamic_weight = dynamic_weight.permute([3, 0, 1, 2])
kernel = weights_flat.view(n_bands, kernel_size, kernel_size, embed_dim)
kernel = kernel.permute(3, 0, 1, 2)   # [embed_dim, n_bands, k, k]
print(f"  Nach view  : {tuple(weights_flat.view(n_bands, kernel_size, kernel_size, embed_dim).shape)}")
print(f"  Nach permute: {tuple(kernel.shape)}")
print(f"  Bedeutung   : (out_channels={embed_dim}, in_channels={n_bands}, k={kernel_size}, k={kernel_size})")
print()
print("  → Das ist ein normaler Conv2d-Kernel — nur eben dynamisch erzeugt statt gelernt.")
print(f"  Zum Vergleich: ein normales ViT hätte einen festen Kernel dieser Form")
print(f"  mit {embed_dim * n_bands * kernel_size * kernel_size:,} Parametern")

## 4. Zusammenführung: conv2d(Bild, Kernel) → Patch-Tokens

In [ ]:
scaler = patch_embed.scaler  # 0.01

print("--- Schritt E: Dynamisches Patch Embedding (conv2d) ---")
print(f"  Bild-Tensor  : {tuple(x.shape)}")
print(f"  Kernel       : {tuple(kernel.shape)}")
print(f"  Bias         : {tuple(bias.shape)}")
print(f"  stride=16, padding=1, scaler={scaler}")
print()

with torch.no_grad():
    k_scaled = kernel * scaler
    b_scaled = bias.view(embed_dim) * scaler
    conv_out = F.conv2d(x, k_scaled, bias=b_scaled, stride=kernel_size, padding=1)

print(f"  Output conv2d: {tuple(conv_out.shape)}")
H_out = conv_out.shape[2]
print(f"  Spatial: ({H}+2×1-{kernel_size})/{kernel_size}+1 = {H_out} → {H_out}×{H_out} = {H_out*H_out} Patches")
print()

# flatten spatial → token sequence
# Code: x = x.flatten(2).transpose(1, 2)
patch_tokens = conv_out.flatten(2).transpose(1, 2)
print(f"  Nach flatten + transpose: {tuple(patch_tokens.shape)}")
print(f"  Bedeutung: (Batch={B}, num_patches={H_out*H_out}, embed_dim={embed_dim})")

## 5. Hauptnetzwerk: Positional Embedding + CLS + Transformer

In [ ]:
print("--- Schritt F: Positional Embedding ---")
print(f"  pos_embed shape    : {tuple(encoder.pos_embed.shape)}")
print(f"  pos_embed[:, 1:]   : [1, {H_out*H_out}, {embed_dim}]  (ohne CLS-Slot)")
print()
tokens = patch_tokens + encoder.pos_embed[:, 1:, :].cpu()
print(f"  patch_tokens + pos : {tuple(tokens.shape)}")
print()

print("--- Schritt G: CLS Token prependen ---")
print(f"  cls_token shape    : {tuple(encoder.cls_token.shape)}")
cls = encoder.cls_token + encoder.pos_embed[:, :1, :]
cls_expanded = cls.expand(B, -1, -1).cpu()
tokens_with_cls = torch.cat([cls_expanded, tokens], dim=1)
print(f"  Nach cat([CLS, tokens]): {tuple(tokens_with_cls.shape)}")
print(f"  Bedeutung: (Batch={B}, {H_out*H_out}+1={H_out*H_out+1} Tokens, {embed_dim} dim)")
print()
print("  → Das ist der Input für alle 12 Transformer-Blöcke")

In [ ]:
print("--- Schritt H: 12× Transformer Block (alle identische Shape) ---")
print(f"  Input  jedes Blocks: {tuple(tokens_with_cls.shape)}")
print(f"  Output jedes Blocks: {tuple(tokens_with_cls.shape)}   (Shape bleibt immer gleich)")
print()
print("  Output-Layer:")
for lay in encoder.output_layers:
    out_h = out_w = H // 16   # 14
    print(f"    Block {lay:2d} → reshape [B, {H_out*H_out}, {embed_dim}] → [B, {embed_dim}, {out_h}, {out_w}]")
print()
print("  reshape-Code:")
print(f"    x[:, 1:]  → entfernt CLS → [B, {H_out*H_out}, {embed_dim}]")
print(f"    .permute(0,2,1) → [B, {embed_dim}, {H_out*H_out}]")
print(f"    .view(B, {embed_dim}, {out_h}, {out_w}) → Feature Map")

## 6. Gesamtübersicht

In [ ]:
print("╔══════════════════════════════════════════════════════════════════╗")
print("║             DOFA — Datenfluss & Tensor-Shapes                    ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  raw_input['optical']         {str((B,C,T,H,W)):>30}  ║")
print(f"║  nach squeeze(2)              {str((B,C,H,W)):>30}  ║")
print("║                                                                  ║")
print("║  ── Hypernetzwerk ──────────────────────────────────────────     ║")
print(f"║  wavelist                     {str((n_bands,)):>30}  ║")
print(f"║  sincos embedding             {str((n_bands, wv_planes)):>30}  ║")
print(f"║  FCResLayer output            {str((n_bands, wv_planes)):>30}  ║")
print(f"║  Transformer-Sequenz          {str((128+n_bands+1, wv_planes)):>30}  ║")
print(f"║  weights (flat)               {str((n_bands, kernel_size*kernel_size*embed_dim)):>30}  ║")
print(f"║  bias                         {str((embed_dim,)):>30}  ║")
print(f"║  conv kernel                  {str((embed_dim, n_bands, kernel_size, kernel_size)):>30}  ║")
print("║                                                                  ║")
print("║  ── Patch Embedding ────────────────────────────────────────     ║")
print(f"║  conv2d output                {str((B, embed_dim, H_out, H_out)):>30}  ║")
print(f"║  patch_tokens                 {str((B, H_out*H_out, embed_dim)):>30}  ║")
print("║                                                                  ║")
print("║  ── Hauptnetzwerk (ViT) ────────────────────────────────────     ║")
print(f"║  + pos_embed + CLS            {str((B, H_out*H_out+1, embed_dim)):>30}  ║")
print(f"║  12× Transformer Block        {str((B, H_out*H_out+1, embed_dim)):>30}  ║")
print(f"║  4 Outputs (je)               {str((B, embed_dim, H_out, H_out)):>30}  ║")
print("╚══════════════════════════════════════════════════════════════════╝")

## 7. Vollständiger Forward-Pass zur Verifikation

In [ ]:
with torch.no_grad():
    outputs = encoder(raw_input)

print("Encoder-Outputs (die 4 Feature Maps die an den Decoder gehen):")
for i, (lay, out) in enumerate(zip(encoder.output_layers, outputs)):
    print(f"  output_layers[{i}] = Block {lay:2d} → {tuple(out.shape)}")

## 8. Visualisierung: Wellenlängen-Embeddings
Zeigt wie ähnlich oder verschieden die 12 S2-Bänder nach der sincos-Kodierung und dem FCResLayer sind.

In [ ]:
with torch.no_grad():
    wv_sincos = get_1d_sincos_pos_embed_from_grid_torch(128, wavelist * 1000).float().cpu()
    wv_fcres  = patch_embed.fclayer(wv_sincos.to(device)).cpu()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# sincos
im0 = axes[0].imshow(wv_sincos.numpy(), aspect="auto", cmap="RdBu_r")
axes[0].set_yticks(range(len(BANDS)))
axes[0].set_yticklabels([f"{b} {wv:.3f}μm" for b,wv in zip(BANDS,wv_list)], fontsize=8)
axes[0].set_xlabel("Dim (128)")
axes[0].set_title("A: sincos Embedding")
plt.colorbar(im0, ax=axes[0])

# FCResLayer output
im1 = axes[1].imshow(wv_fcres.numpy(), aspect="auto", cmap="RdBu_r")
axes[1].set_yticks(range(len(BANDS)))
axes[1].set_yticklabels([f"{b} {wv:.3f}μm" for b,wv in zip(BANDS,wv_list)], fontsize=8)
axes[1].set_xlabel("Dim (128)")
axes[1].set_title("B: Nach FCResLayer")
plt.colorbar(im1, ax=axes[1])

# Cosine similarity
norm = wv_fcres / wv_fcres.norm(dim=1, keepdim=True)
sim  = norm @ norm.T
im2  = axes[2].imshow(sim.numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
axes[2].set_xticks(range(len(BANDS)))
axes[2].set_xticklabels(BANDS, fontsize=8, rotation=45)
axes[2].set_yticks(range(len(BANDS)))
axes[2].set_yticklabels(BANDS, fontsize=8)
axes[2].set_title("C: Cosine Similarity\nzwischen Band-Embeddings")
plt.colorbar(im2, ax=axes[2])

plt.suptitle("Wellenlängen-Embeddings: 12 S2-Bänder (m-cashew-plantation)", fontsize=12)
plt.tight_layout()
plt.show()